# 03. Benchmark Lite tracker di MOT17

Notebook 02 menunjukkan Lite tuned paling baik di CCTV kita, tapi nilainya diukur terhadap tracker referensi, bukan label manusia. Notebook ini menguji Lite di **MOT17**, benchmark publik yang paling umum dipakai paper tracking, supaya hasilnya bisa dibandingkan secara terbuka.

**Yang diukur.** Split train MOT17 (7 video, ada label ID dari manusia). Semua tracker diberi **deteksi publik yang sama** dari MOT17 (FRCNN dan SDP), jadi yang dibandingkan murni kualitas tracking-nya, bukan detektornya. Tidak perlu gambar, hanya file label (sekitar 10 MB).

**Metrik.** Tiga metrik standar MOTChallenge, dihitung dengan `object_tracking.metrics` yang hasilnya sudah dicocokkan sama persis dengan TrackEval resmi:
- **HOTA**: metrik utama MOT17 sekarang. Seimbang antara deteksi (DetA) dan konsistensi ID (AssA).
- **IDF1**: seberapa lama setiap orang memegang ID yang benar. Paling relevan untuk kita.
- **MOTA**: didominasi kualitas deteksi (FP dan FN), kurang peka ke ID.

**Yang perlu diingat saat membaca hasil.**
1. MOT17 adalah pejalan kaki di jalan, 14 sampai 30 FPS, dan 4 dari 7 video kameranya bergerak. CCTV kita diam. Karena itu hasil dipisah antara **kamera diam** (02, 04, 09) dan **kamera bergerak**.
2. Parameter Lite tuned disetel untuk skor detektor D-FINE di CCTV, bukan untuk FRCNN atau SDP. Bagian 5 menyetel ulang ambang secara adil (validasi silang).
3. Angka absolut lebih rendah dari paper karena paper biasanya memakai detektor sendiri yang jauh lebih kuat (misalnya YOLOX). Bandingkan antar tracker di tabel ini, jangan dengan angka paper.
4. Lisensi data MOT17 adalah CC BY-NC-SA 3.0 (non-komersial). Di sini dipakai hanya untuk mengukur, bukan untuk melatih model.

**Cara menjalankan** (sekali): `uv sync`, lalu buka notebook ini di VS Code dengan kernel `.venv`. Untuk cek cepat, set `QUICK = True`.

Semua hasil tersimpan di `experiments/03_mot17/<waktu>/`.

In [ ]:
# Cek Python yang dipakai notebook. Harus .venv milik repo ini.
import sys, importlib.util
print("Python:", sys.executable)
missing = [m for m in ('numpy', 'scipy', 'pandas', 'matplotlib', 'cv2', 'ultralytics') if importlib.util.find_spec(m) is None]
if ".venv" not in sys.executable or missing:
    raise RuntimeError(
        "Notebook belum memakai .venv proyek" + (f" (paket tidak ada: {', '.join(missing)})" if missing else "") + ".\n"
        "Jalankan `uv sync` di folder repo, lalu di VS Code: klik nama kernel di kanan atas > Select Another Kernel > "
        "Python Environments > .venv (object-tracking), atau Enter interpreter path: "
        "~/Documents/GitHub/object-tracking/.venv/bin/python"
    )
print("ok, semua paket ada")

## 0. Pengaturan

In [ ]:
import os, sys, json, time, platform, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))

from object_tracking import mot17
from object_tracking.zoo import PRESETS, LITE_DEFAULT, load_tuned

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

# ===================== ubah di sini =====================
MOT_DIR   = Path(os.environ.get("MOT17_DIR", REPO / "data" / "mot17"))   # tempat label MOT17 diunduh
QUICK     = os.environ.get("OT_QUICK", "0") == "1"   # True = 3 video dan 1 detektor saja, untuk cek notebook jalan
DETECTORS = ["SDP"] if QUICK else ["FRCNN", "SDP"]    # deteksi publik MOT17. DPM tidak dipakai (skornya bukan 0..1)
SEQS      = ["MOT17-02", "MOT17-05", "MOT17-10"] if QUICK else mot17.TRAIN_SEQS
FPS_LIST  = [10, 5, 3, 2]          # FPS proses yang diuji di bagian 4 (selain FPS asli). CCTV kita: 2 sampai 6 FPS
RUN_TUNING = True                  # bagian 5 (validasi silang ambang skor). Sekitar 5 sampai 15 menit di M5
# ========================================================

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + ("_quick" if QUICK else "")
OUT = REPO / "experiments" / "03_mot17" / RUN_ID
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
RESULTS = {"run_id": RUN_ID, "quick": QUICK, "detectors": DETECTORS, "seqs": SEQS,
           "host": {"platform": platform.platform(), "machine": platform.machine()}}

def save_results():
    (OUT / "results.json").write_text(json.dumps(RESULTS, indent=2, default=float, ensure_ascii=False))

def savefig(name):
    plt.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

def prog(x):
    return tqdm(x, leave=False) if tqdm else x

COLS = ["HOTA", "IDF1", "MOTA", "AssA", "DetA", "IDSW", "IDs", "GT_IDs", "ms/frame"]
FMT = {"HOTA": "{:.3f}", "IDF1": "{:.3f}", "MOTA": "{:.3f}", "AssA": "{:.3f}", "DetA": "{:.3f}", "ms/frame": "{:.2f}"}
def show(df, cols=COLS):
    return df[[c for c in cols if c in df.columns]].style.format(FMT).background_gradient(
        subset=[c for c in ("HOTA", "IDF1") if c in df.columns], cmap="Greens")

print("data MOT17:", MOT_DIR)
print("hasil     :", OUT)

## 1. Data MOT17

Sel ini mengunduh `MOT17Labels.zip` (label dan deteksi publik, tanpa gambar) sekali saja. Kalau unduhan gagal, unduh manual dari https://motchallenge.net/data/MOT17Labels.zip lalu taruh zip-nya di folder `data/mot17/`.

In [ ]:
ROOT = mot17.download_labels(MOT_DIR)
seqs = [mot17.load_seq(ROOT, n, d) for d in DETECTORS for n in SEQS]

info = pd.DataFrame([{
    "video": s.name, "FPS": s.fps, "frame": s.length, "durasi (detik)": round(s.length / s.fps, 1),
    "resolusi": f"{s.width}x{s.height}", "orang (ID)": len(np.unique(s.gt[(s.gt[:, 6] != 0) & (s.gt[:, 7] == 1), 1])),
    "kamera": "diam" if s.static else "bergerak",
} for s in seqs if s.detector == DETECTORS[0]]).set_index("video")
display(info)
print("Sebaran skor deteksi publik (Lite tuned memakai ambang YAKIN 0,517 dan ID baru 0,535):")
display(mot17.det_score_summary(seqs).style.format("{:.3f}", subset=["min", "p10", "median", "p90", "max"]))
RESULTS["data"] = info.reset_index().to_dict("records")

## 2. Tracker yang dibandingkan

| Nama | Isi |
|---|---|
| **Lite tuned (CCTV)** | Lite dengan parameter hasil tuning notebook 01, apa adanya |
| **Lite bawaan** | Lite dengan parameter default di `TrackerConfig` |
| **ByteTrack tuned (CCTV)** | ByteTrack Ultralytics dengan parameter hasil tuning notebook 01 |
| **ByteTrack, OC-SORT, BoT-SORT** | Implementasi Ultralytics dengan setelan standar (ambang YAKIN 0,5) |
| **ByteTrack (sekarang)** | Setelan pipeline produksi saat ini: hanya deteksi skor 0,5 ke atas |

BoT-SORT dijalankan tanpa kompensasi gerak kamera dan tanpa Re-ID, sama seperti di notebook 01. Ini merugikannya di video kamera bergerak, jadi lihat juga tabel kamera diam.

In [ ]:
tuned = load_tuned()
SPECS = {}
if "Lite tuned" in tuned:
    SPECS["Lite tuned (CCTV)"] = tuned["Lite tuned"]
SPECS["Lite bawaan"] = LITE_DEFAULT
if "ByteTrack tuned" in tuned:
    SPECS["ByteTrack tuned (CCTV)"] = tuned["ByteTrack tuned"]
for k in ["ByteTrack", "OC-SORT", "BoT-SORT", "ByteTrack (sekarang)"]:
    SPECS[k] = PRESETS[k]
for name, s in SPECS.items():
    print(f"{name:24s} {s.kind:10s} {s.params}")

## 3. FPS asli (standar MOT17)

Cara standar: setiap frame diproses (14 sampai 30 FPS). Ini kondisi yang paling menguntungkan tracker lain, karena mereka dirancang untuk FPS tinggi.

In [ ]:
t0 = time.time()
rows_full = mot17.run_benchmark(SPECS, seqs, fps_list=(None,), progress=prog)
print(f"selesai dalam {time.time() - t0:.0f} detik")

tab_full = mot17.summary(rows_full, by=("tracker",))
RESULTS["fps_asli"] = tab_full.reset_index().to_dict("records")
display(show(tab_full))

**Cara membaca.** Urutan berdasarkan HOTA. IDF1 di sini dinilai terhadap label manusia, jadi bisa dibandingkan langsung. `IDs` yang jauh di atas `GT_IDs` berarti tracker sering membuat ID baru untuk orang yang sama.

Hasil per detektor (FRCNN lebih lemah, SDP lebih kuat):

In [ ]:
tab_det = mot17.summary(rows_full, by=("detektor", "tracker"), sort=None)
display(tab_det[["HOTA", "IDF1", "MOTA", "IDSW"]].unstack(0).style.format("{:.3f}", subset=pd.IndexSlice[:, ["HOTA", "IDF1", "MOTA"]]))

### Kamera diam vs bergerak

Video kamera diam (02, 04, 09) paling mirip CCTV kita. Di video kamera bergerak, tracker tanpa kompensasi gerak kamera (termasuk Lite) wajar turun.

In [ ]:
tab_cam = mot17.summary(rows_full, by=("kamera", "tracker"), sort=None)
RESULTS["fps_asli_per_kamera"] = tab_cam.reset_index().to_dict("records")
pv = tab_cam[["HOTA", "IDF1", "IDSW"]].unstack(0)
display(pv.sort_values(("HOTA", "diam"), ascending=False).style.format("{:.3f}", subset=pd.IndexSlice[:, ["HOTA", "IDF1"]]))

## 4. FPS rendah (kondisi CCTV kita)

Supaya 12 kamera muat di satu mesin, tiap kamera hanya diproses 2 sampai 6 FPS. Di sini video MOT17 diproses dengan melompati frame, lalu dinilai hanya di frame yang diproses. Di kondisi inilah Kalman berbasis detik dan box diperbesar pada Lite seharusnya paling berpengaruh.

In [ ]:
t0 = time.time()
rows_low = mot17.run_benchmark(SPECS, seqs, fps_list=FPS_LIST, progress=prog)
print(f"selesai dalam {time.time() - t0:.0f} detik")
rows_all = rows_full + rows_low

def fps_table(rows, metric, kamera=None):
    r = [x for x in rows if kamera is None or x["kamera"] == kamera]
    t = mot17.summary(r, by=("tracker", "target_fps"), sort=None)[metric].unstack(1)
    t = t[["asli"] + [f for f in FPS_LIST if f in t.columns]]
    return t.sort_values(FPS_LIST[-1] if FPS_LIST[-1] in t.columns else "asli", ascending=False)

for metric in ("HOTA", "IDF1"):
    for kam in (None, "diam"):
        t = fps_table(rows_all, metric, kam)
        RESULTS[f"fps_{metric}_{kam or 'semua'}"] = t.reset_index().to_dict("records")
        print(f"{metric}, {'semua video' if kam is None else 'kamera diam saja'} (kolom = FPS proses)")
        display(t.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, kam in zip(axes, (None, "diam")):
    t = fps_table(rows_all, "IDF1", kam)
    xs = ["asli"] + FPS_LIST
    for name, row in t.iterrows():
        lite = name.startswith("Lite")
        ax.plot(range(len(xs)), [row[x] for x in xs], marker="o", lw=2.5 if lite else 1.2,
                ls="-" if lite else "--", label=name)
    ax.set_xticks(range(len(xs)), ["asli\n(14-30)"] + [str(x) for x in FPS_LIST])
    ax.set_xlabel("FPS proses per kamera")
    ax.set_title("Semua video" if kam is None else "Kamera diam saja")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("IDF1")
axes[1].legend(fontsize=8, loc="lower left")
plt.suptitle("IDF1 di MOT17 train saat FPS diturunkan")
plt.tight_layout()
savefig("idf1_vs_fps")
save_results()

## 5. Penyesuaian ambang yang adil (validasi silang)

Ambang skor Lite tuned disetel untuk D-FINE, sedangkan skor FRCNN dan SDP sebarannya berbeda. Supaya perbandingan adil, Lite dan ByteTrack sama-sama diberi pencarian kecil untuk ambang skor dan IoU:

- Video dibagi dua kelompok. Parameter dipilih di kelompok A lalu dinilai di kelompok B, dan sebaliknya. Jadi tidak ada parameter yang dinilai di video tempat ia dipilih.
- Kriteria pemilihan: rata-rata HOTA di FPS asli dan 5 FPS.
- Parameter lain Lite (OCR, ORU, buffer, waktu hilang) tetap seperti hasil tuning notebook 01.

In [ ]:
FOLD_A = ["MOT17-02", "MOT17-05", "MOT17-09", "MOT17-11"]
FOLD_B = ["MOT17-04", "MOT17-10", "MOT17-13"]
TUNE_FPS = (None, 5)
SCORE_PAIRS = [(0.4, 0.45), (0.5, 0.55), (0.6, 0.65), (0.7, 0.75)]   # (ambang YAKIN, ambang ID baru)

def lite_grid(base):
    for hi, new in SCORE_PAIRS:
        for iou in (0.2, 0.3, 0.4):
            yield f"YAKIN {hi}, baru {new}, IoU {iou}", base.with_params(high_thresh=hi, new_thresh=new, min_iou=iou)

def byte_grid(base):
    for hi, new in SCORE_PAIRS:
        for match in (0.7, 0.8, 0.9):            # Ultralytics: batas biaya 1 - IoU
            yield f"YAKIN {hi}, baru {new}, match {match}", base.with_params(
                track_high_thresh=hi, new_track_thresh=new, match_thresh=match)

FAMILIES = {}
if "Lite tuned (CCTV)" in SPECS:
    FAMILIES["Lite (disetel di MOT17)"] = lite_grid(SPECS["Lite tuned (CCTV)"])
FAMILIES["ByteTrack (disetel di MOT17)"] = byte_grid(PRESETS["ByteTrack"])

cv_rows, cv_pick = [], []
if RUN_TUNING:
    t0 = time.time()
    for fam, grid in FAMILIES.items():
        grid = list(grid)
        for train, test in ((FOLD_A, FOLD_B), (FOLD_B, FOLD_A)):
            tr_seqs = [s for s in seqs if s.name in train]
            te_seqs = [s for s in seqs if s.name in test]
            if not tr_seqs or not te_seqs:
                continue
            scores = []
            for label, spec in prog(grid):
                r = mot17.run_benchmark({label: spec}, tr_seqs, fps_list=TUNE_FPS)
                scores.append((mot17.summary(r, by=("target_fps",), sort=None)["HOTA"].mean(), label, spec))
            best = max(scores, key=lambda x: x[0])
            cv_pick.append({"tracker": fam, "dipilih di": "+".join(n[-2:] for n in train), "parameter": best[1],
                            "HOTA saat memilih": best[0]})
            spec = best[2].with_params()
            spec.name = fam
            cv_rows += mot17.run_benchmark({fam: spec}, te_seqs, fps_list=(None,) + tuple(FPS_LIST))
    print(f"selesai dalam {time.time() - t0:.0f} detik")
    display(pd.DataFrame(cv_pick))
else:
    print("RUN_TUNING = False, bagian ini dilewati")

In [ ]:
if cv_rows:
    # pembanding: tracker bawaan dinilai di video yang sama dengan hasil validasi silang (semua video yang ada di fold)
    in_folds = set(FOLD_A) | set(FOLD_B)
    base_rows = [r for r in rows_all if r["seq"] in in_folds]
    rows_cv = base_rows + cv_rows
    for kam in (None, "diam"):
        t = fps_table(rows_cv, "HOTA", kam)
        RESULTS[f"cv_HOTA_{kam or 'semua'}"] = t.reset_index().to_dict("records")
        print(f"HOTA setelah validasi silang, {'semua video' if kam is None else 'kamera diam saja'}")
        display(t.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0))
    t = fps_table(rows_cv, "IDF1")
    RESULTS["cv_IDF1_semua"] = t.reset_index().to_dict("records")
    print("IDF1 setelah validasi silang, semua video")
    display(t.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0))
    RESULTS["cv_pilihan"] = cv_pick
    save_results()

## 6. Kecepatan tracker

Waktu tracker saja per frame (tanpa detektor), di CPU satu core. Untuk 12 kamera pada 4 FPS, tracker dipanggil 48 kali per detik.

In [ ]:
spd = mot17.summary(rows_full, by=("tracker",), sort=None)[["ms/frame"]].sort_values("ms/frame")
spd["ms per detik, 12 kamera x 4 FPS"] = spd["ms/frame"] * 48
RESULTS["kecepatan"] = spd.reset_index().to_dict("records")
display(spd.style.format("{:.2f}"))
print("Catatan: jumlah orang per frame di MOT17 (sekitar 10 sampai 50) lebih banyak dari CCTV gudang, jadi angka ini batas atas.")

## 7. Simpan hasil

In [ ]:
flat = pd.DataFrame([{k: v for k, v in r.items() if k != "raw"} | mot17.mm.summarize(r["raw"]) for r in rows_all + cv_rows])
flat.to_csv(OUT / "per_video.csv", index=False)
save_results()
print("tersimpan:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(REPO))

## 8. Video perbandingan: video asli vs semua tracker

Satu video MOT17 dirender jadi **grid**: video asli, label manusia (ground truth), lalu satu panel per tracker dari bagian 2. Semua tracker memakai deteksi publik yang sama, jadi perbedaan antar panel murni dari tracker.

Cara membaca tiap panel tracker:
- **warna + nomor** = ID dari tracker. Orang yang sama seharusnya memegang warna yang sama sepanjang video.
- **garis** = jejak gerak 2 detik terakhir.
- **tepi MERAH tebal "SWITCH"** = ID switch: orang yang sama menurut label manusia tiba-tiba memegang ID tracker lain. Tanda ini tampil 1 detik.
- **abu-abu** = box di orang yang tidak dinilai MOT17 (orang di kendaraan, pantulan, distraktor).
- header: IDF1 dan HOTA untuk video ini, **ID switch sejauh ini / total** (totalnya sama dengan kolom IDSW di tabel), dan jumlah ID yang sudah dibuat.

Video mentah diunduh sekali dari MOTChallenge (sekitar 5 MB per video, setengah resolusi) ke `data/mot17/videos/`. Pilihan video kamera diam yang paling mirip CCTV: `MOT17-02`, `MOT17-04`, `MOT17-09`.

In [ ]:
from IPython.display import Image as IPImage, Video
from object_tracking import mot17_video

# ===================== ubah di sini =====================
VIDEO_SEQ   = "MOT17-04"   # MOT17-02 / 04 / 09 = kamera diam (mirip CCTV); 05 / 10 / 11 / 13 = kamera bergerak
VIDEO_DET   = "SDP"        # deteksi publik: "SDP" (kuat) atau "FRCNN" (lebih lemah, lebih banyak ID switch)
VIDEO_FPS   = 5            # FPS proses tracker, seperti CCTV kita (2 sampai 6). None = FPS asli video
VIDEO_MAX_SEC = None       # potong durasi (detik). None = seluruh video
PANEL_WIDTH = 640          # lebar tiap panel (piksel). 480 = file lebih kecil
SHOW_GT     = True         # tampilkan panel label manusia
EMBED_VIDEO = True         # putar video langsung di notebook (ukuran notebook bertambah sebesar file video)
# ========================================================

vseq = mot17.load_seq(ROOT, VIDEO_SEQ, VIDEO_DET)
vpath = mot17_video.download_video(ROOT, VIDEO_SEQ)
fps_tag = "asli" if not VIDEO_FPS else f"{VIDEO_FPS}fps"
t0 = time.time()
vid = mot17_video.render_comparison(
    vseq, vpath, SPECS, OUT / "videos" / f"{VIDEO_SEQ}-{VIDEO_DET}_{fps_tag}_perbandingan.mp4",
    target_fps=VIDEO_FPS, panel_width=PANEL_WIDTH, show_gt=SHOW_GT, max_sec=VIDEO_MAX_SEC, progress=prog)
print(f"selesai dalam {time.time() - t0:.0f} detik: {vid['size'][0]}x{vid['size'][1]}, {vid['fps']:.1f} FPS")
print("video   :", vid["video"].relative_to(REPO), f"({vid['video'].stat().st_size / 1e6:.1f} MB)")
print("cuplikan:", vid["snapshot"].relative_to(REPO))

tab_vid = pd.DataFrame(vid["table"]).set_index("tracker").sort_values("IDF1", ascending=False)
RESULTS[f"video_{VIDEO_SEQ}_{VIDEO_DET}_{fps_tag}"] = tab_vid.reset_index().to_dict("records")
save_results()
print(f"\nAngka untuk video ini saja ({VIDEO_SEQ}, {VIDEO_DET}, {fps_tag}). GT_IDs = jumlah orang menurut label")
display(tab_vid.style.format("{:.3f}", subset=["IDF1", "HOTA", "MOTA"]).background_gradient(cmap="Greens", subset=["IDF1", "HOTA"]))

In [ ]:
# Cuplikan di tengah video, lalu videonya
display(IPImage(filename=str(vid["snapshot"]), width=1100))
if EMBED_VIDEO:
    display(Video(str(vid["video"]), embed=True, width=1100, html_attributes="controls loop"))
else:
    print("buka file video di atas dengan pemutar video (QuickTime, VLC, dll.)")

Untuk video CCTV sendiri (tanpa label), pakai aplikasi di notebook `04_bandingkan_tracker_video.ipynb`.

## Cara menarik kesimpulan

Pertanyaan yang ingin dijawab notebook ini, dan tabel yang menjawabnya:

1. **Apakah Lite setara tracker publik di kondisi standar?** Lihat bagian 3. Selisih HOTA atau IDF1 di bawah sekitar 0,01 bisa dianggap setara.
2. **Apakah Lite lebih baik di FPS rendah, seperti di CCTV kita?** Lihat bagian 4, kolom 2 sampai 5 FPS, terutama tabel kamera diam.
3. **Apakah keunggulannya bertahan setelah semua tracker disetel adil?** Lihat bagian 5. Kalau Lite masih unggul di FPS rendah setelah validasi silang, keunggulannya berasal dari desain (Kalman berbasis detik, OCR, ORU), bukan dari tuning ke data kita.
4. **Di mana Lite lemah?** Video kamera bergerak (05, 10, 11, 13). Ini wajar karena Lite tidak punya kompensasi gerak kamera, dan tidak relevan untuk CCTV yang diam.

Batasan: MOT17 berbeda dari gudang kita (jenis orang, kepadatan, sudut kamera, detektor). Hasil di sini adalah bukti pendukung dari benchmark publik, bukan pengganti evaluasi dengan label ID manual di klip CCTV kita sendiri.